<h1>Section 4 – Language Models exploration</h1>
In this task, the objective is to harness the capabilities of language models like Bert or Word2Vec, for
supervised learning.
Two interesting concepts play a role when we use neural networks:
1- Itis possible to do transfer learning, i.e., to take a model that have been trained with other
enormous datasets by Big Tech companies, and we can do fine-tuning i.e., to train this model
starting from its pre-trained version.
2- In NLP tasks, words/documents are transformed into vectors (encoding) and this task is
Unsupervised, so we can use a much larger amount of data.
Choose a language model between Bert and Doc2Vec (word2vec for documents), then:
<h2>Perform text embedding on comments</h2>
i.e., body column in data_unsupervised.csv.

<s>a. If you choose Doc2Vec, pretrain Doc2Vec based on the comments. Save the embedding
vectors or model for next steps.</s>

b. If you choose Bert, take the pretrained Bert model like in this example from HuggingFace.
You only need the embedding vectors from the model rather than other down-stream task
capabilities, so just take the base model and extract the embeddings for the following
steps. And you are not required to do fine-tuning here. You can use the Bert, DistillBert
(light version of Bert, smaller, faster), or any other Bert variations.

In [6]:
!pip install transformers
!pip install torch


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [7]:
import torch
#from transformers import BertTokenizer, BertModel
from transformers import DistilBertTokenizer
# OPTIONAL: if you want to have more information on what's happening, activate the logger as follows
#import logging
#logging.basicConfig(level=logging.INFO)

import matplotlib.pyplot as plt
#% matplotlib inline

# Load pre-trained model tokenizer (vocabulary)
#tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
#distilbert smaller but faster model
#uncased -> not case-sensitive

import pandas as pd
data = pd.read_csv('data_unsupervised.csv')[:1000]

In [8]:
from transformers import DistilBertModel

# Load pre-trained DistilBert model (outputs hidden states)
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model.eval()

# Function to get embeddings for a list of texts
def get_embeddings(texts, tokenizer, model, max_length=128):
    embeddings = []
    with torch.no_grad():
        count=0 # Just to understand how much time this takes
        for text in texts:
            inputs = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=max_length)
            outputs = model(**inputs)
            # Use the [CLS] token embedding as sentence embedding
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
            embeddings.append(cls_embedding)
            count += 1
            if count % 10000 == 0:
                print(f"Processed {count} comments")
    return embeddings

# Get embeddings for all comments in 'body'
embeddings = get_embeddings(data['body'].tolist(), tokenizer, model)

<h2>Feature Concatenation.</h2>
For each user, create a feature vector where each column x represents the fraction of times the
user has posted in the x-th subreddit (and reduce the cardinality of the matrix with PCA).
Concatenate the embedding vectors output by the model you chose at step a with these new
features.

In [ ]:
subreddits = data['subreddit'].unique()
user_subreddit_counts = data.groupby(['author', 'subreddit']).size().unstack(fill_value=0)
user_total_comments = user_subreddit_counts.sum(axis=1)
user_sub_fraction = user_subreddit_counts.div(user_total_comments, axis=0)
user_sub_fraction = user_sub_fraction.reindex(columns=subreddits, fill_value=0)

# user_sub_fraction is a DataFrame where rows are users and columns are subreddits,
# and values are the fraction of comments in each subreddit for each user
user_sub_fraction.head()

In [ ]:
from sklearn.decomposition import PCA

pca=PCA(random_state=42,)
pca=pca.fit(user_sub_fraction)
X_pca=pca.transform(user_sub_fraction)

In [ ]:
concatenated_data = pd.concat([X_pca, embeddings], axis=1)